In [ ]:
!pip install -U qwen-tts

In [ ]:
import gradio as gr
import torch
import soundfile as sf
import numpy as np

from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel

# 1. 현재 환경 자동 감지 (Colab vs Local)
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# 2. 하드웨어 가속 및 데이터 타입 자동 설정
if torch.cuda.is_available():
    device = "cuda:0"
    dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

print(f"🌍 현재 환경: {'Google Colab' if IS_COLAB else 'Local'}")
print(f"💻 로드된 디바이스: {device} (dtype: {dtype})")

# 3. 모델 로드
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"
model = Qwen3TTSModel.from_pretrained(model_id, device_map=device, dtype=dtype)

def generate_audio(text, lang):
    output_path = "temp_audio.wav"
    speaker = "Sohee"
    wavs, sr = model.generate_custom_voice(
        text=text,
        language=lang,
        speaker=speaker,
    )

    sf.write(output_path, wavs[0], sr)

    return output_path

# Gradio 인터페이스 구성
demo = gr.Interface(
    fn=generate_audio,
    inputs=[
        gr.Textbox(label="Text (합성할 텍스트)", placeholder="여기에 대본을 넣으세요..."),
        gr.Dropdown(label="Language", choices=["Korean", "Chinese", "English"], value="Korean")
    ],
    outputs=gr.Audio(label="Output Audio"),
    title="Qwen3-TTS Custom Voice Studio (Colab Edition)"
)

# 5. 환경에 따른 실행 옵션 자동 분기
if IS_COLAB:
    # 코랩: 외부 브라우저에서 접속하기 위해 share=True (터널링) 필수
    print("🚀 코랩 환경이 감지되었습니다. Public URL을 생성합니다.")
    demo.launch(share=True, debug=True)
else:
    # 로컬: 8000번 포트 고정, 내부 네트워크에서만 접근 (보안 유지 및 속도 빠름)
    print("🏠 로컬 환경이 감지되었습니다. http://localhost:8000 으로 접속하세요.")
    demo.launch(server_name="0.0.0.0", server_port=8000, share=False, debug=True)

In [ ]:
import gradio as gr
import torch
import soundfile as sf
import numpy as np
import os

from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel
from huggingface_hub import scan_cache_dir

def check_model_local(repo_id):
    try:
        # 허깅페이스 로컬 캐시 디렉토리를 스캔해서 해당 모델이 있는지 확인
        hf_cache_info = scan_cache_dir()
        for repo in hf_cache_info.repos:
            if repo.repo_id == repo_id:
                return True
        return False
    except Exception as e:
        print(f"캐시 스캔 중 오류: {e}")
        return False

# 2. 모델 존재 여부에 따른 오프라인/온라인 분기
if check_model_local(model_id):
    print(f"✅ 모델 '{model_id}'가 로컬에 존재합니다. 오프라인 모드로 전환합니다.")
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    local_files_only = True
else:
    print(f"🌐 모델 '{model_id}'가 로컬에 없습니다. 다운로드를 시작합니다 (네트워크 필요).")
    os.environ['HF_HUB_OFFLINE'] = '0'
    os.environ['TRANSFORMERS_OFFLINE'] = '0'
    local_files_only = False

# 1. 현재 환경 자동 감지 (Colab vs Local)
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# 2. 하드웨어 가속 및 데이터 타입 자동 설정
if torch.cuda.is_available():
    device = "cuda:0"
    dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

print(f"🌍 현재 환경: {'Google Colab' if IS_COLAB else 'Local'}")
print(f"💻 로드된 디바이스: {device} (dtype: {dtype})")

# 3. 모델 로드
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
model = Qwen3TTSModel.from_pretrained(model_id, device_map=device, dtype=dtype, local_files_only=True)
prompt_file_path = "./voice-data/voice_prompt.pt"

# 4. Voice 데이터 가져오기
if os.path.exists(prompt_file_path):
    # 파일이 있으면 저장된 목소리 특징 불러오기
    print("저장된 목소리 특징 파일을 불러옵니다...")
    prompt_items = torch.load(prompt_file_path, map_location=device) 
else:
    # 파일이 없으면 새로 추출하고 파일로 저장하기
    print("목소리 특징 파일이 없습니다. 새로 추출을 시작합니다...")
    prompt_items = model.create_voice_clone_prompt(
        ref_audio="./audio-ref/dm_audio2.wav",
        ref_text="안녕하세요! 매일 쏟아지는 최신 기술 트렌드를 알기 쉽게 전해드리는 시간입니다. \
오늘은 최근 웹 생태계에서 화제가 되고 있는 새로운 AI 최적화 방식에 대해 알아볼까 해요. \
다들 프로젝트를 배포하시면서 서버 성능 문제로 한 번쯤은 답답함을 느껴보셨을 텐데요. \
이게 바로 우리가 새로운 아키텍처에 주목해야 하는 가장 큰 이유인 것이죠. \
이 기술을 실제 환경에 도입해 보면, 전반적인 시스템 속도가 놀라울 정도로 개선되는 것을 직접 체감하실 수 있습니다. \
자, 그럼 어떤 원리로 작동하는지 지금부터 함께 살펴보겠습니다!",
        x_vector_only_mode=False,
    )
    # 추출한 특징을 파일로 영구 저장
    torch.save(prompt_items, prompt_file_path)
    print(f"목소리 특징이 {prompt_file_path}에 저장되었습니다!")

def generate_audio(text, lang):
    output_path = "./temp_audio.wav"

    wavs, sr = model.generate_voice_clone(
        text=text,
        language=lang,
        voice_clone_prompt=prompt_items
    )

    wav = wavs[0]
    silence = np.zeros(int(sr * 0.5), dtype=wav.dtype)
    wav = np.concatenate([wav, silence])

    sf.write(output_path, wav, sr)

    return output_path

# 5. Gradio 인터페이스 구성
demo = gr.Interface(
    fn=generate_audio,
    inputs=[
        gr.Textbox(label="Text (합성할 텍스트)", placeholder="여기에 대본을 넣으세요..."),
        gr.Dropdown(label="Language", choices=["Korean", "Chinese", "English"], value="Korean")
    ],
    outputs=gr.Audio(label="Output Audio"),
    title=f"Qwen3-TTS Voice Design Studio ({'Colab' if IS_COLAB else 'Local'} Edition)"
)

# 6. 환경에 따른 실행 옵션 자동 분기
if IS_COLAB:
    # 코랩: 외부 브라우저에서 접속하기 위해 share=True (터널링) 필수
    print("🚀 코랩 환경이 감지되었습니다. Public URL을 생성합니다.")
    demo.launch(share=True, debug=True)
else:
    # 로컬: 8000번 포트 고정, 내부 네트워크에서만 접근 (보안 유지 및 속도 빠름)
    print("🏠 로컬 환경이 감지되었습니다. http://localhost:8000 으로 접속하세요.")
    demo.launch(server_name="0.0.0.0", server_port=8000, share=False, debug=True)

ImportError: cannot import name 'scan_cache_repo' from 'huggingface_hub' (/Users/kdm/miniconda3/envs/qwen3-tts/lib/python3.12/site-packages/huggingface_hub/__init__.py)